## 1. Libraries
`importlib.reload` makes edits in `strava_data/` pick up without restarting the kernel.

In [ ]:
import importlib
import logging
import pandas as pd
import numpy as np

import strava_data
import strava_data.authentication
import strava_data.visualization
import strava_data.activity_cache
importlib.reload(strava_data)
importlib.reload(strava_data.authentication)
importlib.reload(strava_data.visualization)
importlib.reload(strava_data.activity_cache)

from strava_data.authentication import login
import strava_data.visualization as vis
# Disk-cached per-activity fetches keyed by id — only new activities hit the API,
# keeping re-runs well under Strava's short-term rate limit. Delete .cache/*.json to refresh.
from strava_data.activity_cache import fetch_text_fields, fetch_velocity_streams

# Show plots inline in the notebook (update_plots.py sets this to False for CI)
vis.SHOW_PLOTS = True

# Silence stravalib's per-request 'No rates present in response headers' warning
logging.getLogger('stravalib.util.limiter').setLevel(logging.ERROR)

## 2. Login
Uses `secrets/client_secrets.txt` + `secrets/strava_token.json` locally, or `STRAVA_CLIENT_*` env vars in CI.

In [ ]:
client = login(secrets_folder="../secrets")

## 3. Get activities
Pulls the most recent 1000 activities and unpacks the columns we use downstream.

In [ ]:
activities_object = client.get_activities(limit=1000)
activities = list(activities_object)

ACTIVITY_COLS = [
    'id', 'type', 'name', 'distance', 'moving_time', 'elapsed_time',
    'total_elevation_gain', 'start_date', 'start_latlng', 'kilojoules',
    'average_heartrate', 'max_heartrate', 'elev_high', 'elev_low',
    'average_speed', 'max_speed',
]

def get_activity_data(activity):
    activity_dict = dict(activity)
    row = {k: activity_dict[k] for k in ACTIVITY_COLS}
    # sport_type distinguishes trail runs (type is the legacy 'Run' for both);
    # start_date_local gives the correct calendar day. Both optional in summary payload.
    row['sport_type'] = activity_dict.get('sport_type')
    row['start_date_local'] = activity_dict.get('start_date_local')
    return row

df_activities = pd.DataFrame([get_activity_data(a) for a in activities])
df_activities.head()

## 4. Prepare run data
Filter to runs from 2025+ and add the derived columns the plotting code expects:
- `distance_km` — distance in km (Strava returns meters)
- `week` — end-of-week timestamp (Sunday) for grouping

In [ ]:
df_activities['start_date'] = pd.to_datetime(df_activities['start_date'], utc=True)

df_runs = df_activities[
    (df_activities['type'] == 'Run') &
    (df_activities['start_date'].dt.year >= 2025)
].copy()

df_runs['distance_km'] = df_runs['distance'] / 1000
df_runs['week'] = df_runs['start_date'].dt.to_period('W-SUN').apply(lambda r: r.end_time)

print(f'{len(df_runs)} runs from {df_runs["start_date"].min().date()} to {df_runs["start_date"].max().date()}')
df_runs.head()

## 5. Weekly aggregates
Sum volume + longest run per week. Make sure the current week is in the index even if no runs have happened yet.

In [ ]:
df_runs_weekly = df_runs.groupby('week')['distance_km'].agg(
    total_volume='sum',
    long_run='max',
).sort_index()

now = pd.Timestamp.now(tz=df_runs['start_date'].dt.tz)
this_week = now.to_period('W-SUN').end_time

if this_week not in df_runs_weekly.index:
    df_runs_weekly.loc[this_week] = {'total_volume': 0, 'long_run': 0}
df_runs_weekly = df_runs_weekly.sort_index()

df_runs_weekly.tail(8)

## 6. Target calculation
Base = max volume over the last 4 *completed* weeks. The weekly increase is **+25%** while a boosted target stays below the **recovery ceiling** (the highest 3-consecutive-week average volume of the last half year — a previously sustained level); once +25% would reach it, growth reverts to **+10%**. See `vis.grow_target`.

If this week is already past the target, bump the target by another step and flag that planning should target **next** week.

In [ ]:
# Last 4 completed weeks (exclude this week)
completed_weeks = df_runs_weekly.loc[df_runs_weekly.index < this_week]
recent_completed = completed_weeks.iloc[-4:]
base_completed = recent_completed['total_volume'].max() if len(recent_completed) > 0 else 0

# Recovery ceiling: highest mean of any 3 consecutive recorded weeks over the last
# half year (~26 weeks) = a previously sustained volume. Build-up grows at +25%/week
# while staying under it; once a +25% step would reach it, growth reverts to +10%.
half_year = completed_weeks.iloc[-26:]
recovery_ceiling = half_year['total_volume'].rolling(3).mean().max()
recovery_ceiling = None if pd.isna(recovery_ceiling) else float(recovery_ceiling)

this_week_volume = df_runs_weekly.loc[this_week, 'total_volume']
this_week_target = vis.grow_target(base_completed, recovery_ceiling)
target_reached = this_week_volume >= this_week_target

# week_target is this week's effective level: the goal while building toward it, or the
# achieved volume once reached. It's the base the progression plot grows future weeks from.
week_target = this_week_target if not target_reached else max(this_week_volume, this_week_target)
week_target = round(float(week_target), 1)
week_ran = round(float(this_week_volume), 1)

# Once this week's target is reached, this week is done — plan next week instead, using a
# target grown one step from this week's level.
target_next_week = target_reached
plan_target = round(vis.grow_target(week_target, recovery_ceiling), 1) if target_next_week else week_target

print(f'base (max of last 4 completed weeks): {base_completed:.1f} km')
print(f'recovery ceiling (max 3-wk mean, 26w): {recovery_ceiling if recovery_ceiling is None else round(recovery_ceiling, 1)} km')
print(f'this week ran / target:              {week_ran:.1f} / {this_week_target:.1f} km  (reached={target_reached})')
print(f'effective week_target:               {week_target:.1f} km')
print(f'plan next week instead?              {target_next_week}')
print(f'plan target ({"next" if target_next_week else "this"} week):              {plan_target:.1f} km')

## 7. Weekly stacked plots (runs)
Each bar is one week, each stack segment is one run. Color encodes distance, pace, risk (z-score combining distance and speed), or upper-quartile speed (75th-percentile of the per-run velocity stream).

In [ ]:
vis.plot_weekly(df_runs, col='distance', save_name='weekly_distance.png')
vis.plot_weekly(df_runs, col='pace', save_name='weekly_pace.png')
vis.plot_weekly(df_runs, col='risk', save_name='weekly_risk.png')

In [ ]:
# Color = N-th percentile speed (km/h) per run.
# Tweak PERCENTILE freely — streams are cached on disk, so only the colormap recomputes.
#   75 = upper quartile (top 25%) ·  90 = top decile  ·  95 = top 5%  ·  99 = top 1%
PERCENTILE = 80

# Disk-cached velocity streams (strava_data/activity_cache.py): only new runs hit the API.
# A full first fetch (~1 request/run) may trip the rate limit — it stops early and
# resumes from cache on the next run, so just re-run this cell until it's all cached.
streams = fetch_velocity_streams(client, df_runs['id'].tolist())

def percentile_speed_kmh(activity_id, percentile):
    data = streams.get(activity_id)
    if data is None or len(data) == 0:
        return float('nan')
    return float(np.percentile(data, percentile)) * 3.6  # m/s → km/h

col_name = f'p{PERCENTILE}_speed_kmh'
df_runs[col_name] = df_runs['id'].apply(lambda aid: percentile_speed_kmh(aid, PERCENTILE))

vis.plot_weekly_stacked(
    df_runs.dropna(subset=[col_name]),
    stack_col='distance_km',
    color_col=col_name,
    stack_label='Distance (km)',
    color_label=f'P{PERCENTILE} speed (km/h)',
    title=f'Weekly Distance Stacked per Run  |  P{PERCENTILE} Speed',
    save_name=f'weekly_p{PERCENTILE}_speed.png',
)

## 8. Weekly target progression
Past weeks (white) + this week's progress (white done, orange remaining) + future growth targets (orange). The dark-gray dashed line is the recovery ceiling: below it growth is +25%/week, at/above it +10%.

In [ ]:
vis.plot_weekly_distance_targets(
    df_runs_weekly,
    week_target=week_target,
    this_week=this_week,
    this_week_target=this_week_target,
    this_week_volume=this_week_volume,
    target_reached=target_reached,
    recovery_ceiling=recovery_ceiling,
    additional_weeks=4,
    last_weeks=7,
    save_name='weekly_distance_targets.png',
)

## 9. Example week plans
Idealized split of `plan_target` km across 3, 4, or 5 runs using fixed proportions. When this week's target is already reached, `plan_target` is next week's grown target so these examples match the next-week plan below.

In [ ]:
for runs in [3, 4, 5]:
    vis.plot_week_plan(plan_target, runs, save_name=f'week_plan_{runs}_runs.png')

## 10. Current / next week plan
Already-ran days in white, proposed remaining runs in orange. Switches to *next* week automatically if this week's target is already passed (`target_next_week=True`).

In [ ]:
for runs in [3, 4, 5]:
    vis.plot_current_week_plan(
        df_runs,
        plan_target,
        runs=runs,
        target_next_week=target_next_week,
        save_name=f'current_week_plan_{runs}_runs.png',
    )

## 11. Hiking — prepare data
Filter `Hike` activities from 2025+. Strava's summary API doesn't return `private_note`, so we call `get_activity` per hike to pick up description + private note — but results are cached on disk per id (`strava_data/activity_cache.py`), so only *new* hikes hit the API. Then regex-parse `NNkg` from title / description / private note. Hikes without a reported kg get `weight_kg = 0`.

In [ ]:
import re

KG_PATTERN = re.compile(r'(\d+)\s*kg', re.IGNORECASE)

def parse_weight_kg(*texts):
    """Return the first 'NN kg' int found across the given text fields, else 0."""
    for t in texts:
        if not isinstance(t, str) or not t:
            continue
        m = KG_PATTERN.search(t)
        if m:
            return int(m.group(1))
    return 0

df_hikes = df_activities[
    (df_activities['type'] == 'Hike') &
    (df_activities['start_date'].dt.year >= 2025)
].copy()

df_hikes['distance_km'] = df_hikes['distance'] / 1000
df_hikes['week'] = df_hikes['start_date'].dt.to_period('W-SUN').apply(lambda r: r.end_time)

# Disk-cached fetch (strava_data/activity_cache.py): only new hikes hit the API.
hike_text = fetch_text_fields(client, df_hikes['id'].tolist())
df_hikes = df_hikes.merge(hike_text, on='id', how='left')
df_hikes['weight_kg'] = df_hikes.apply(
    lambda r: parse_weight_kg(r['name'], r.get('description'), r.get('private_note')),
    axis=1,
)

print(f'{len(df_hikes)} hikes; {(df_hikes["weight_kg"] > 0).sum()} with reported kg')
df_hikes[['start_date', 'name', 'distance_km', 'weight_kg']].head()

## 12. Hiking weekly stacked plot
Each bar = one week. Each stack segment = one hike — height is `distance_km`, color is `weight_kg` (carried weight; 0 if not reported).

In [ ]:
vis.plot_weekly_stacked(
    df_hikes,
    stack_col='distance_km',
    color_col='weight_kg',
    stack_label='Distance (km)',
    color_label='Carried weight (kg)',
    title='Weekly Hiking Distance  |  Carried Weight',
    save_name='weekly_hike_weight.png',
)

## 13. Strength training — prepare data
Filter `WeightTraining` activities from 2025+. Parse `NNNN kg volume` from description / private note → `volume_kg`. Sessions where volume isn't logged (early entries) are dropped. `time_min` = moving time in minutes.

In [ ]:
VOLUME_PATTERN = re.compile(r'(\d{2,7})\s*kg\s*volume', re.IGNORECASE)

def parse_volume_kg(*texts):
    """Return the int from 'NNNN kg volume' across the given text fields, else None."""
    for t in texts:
        if not isinstance(t, str) or not t:
            continue
        m = VOLUME_PATTERN.search(t)
        if m:
            return int(m.group(1))
    return None

df_strength = df_activities[
    (df_activities['type'] == 'WeightTraining') &
    (df_activities['start_date'].dt.year >= 2025)
].copy()

df_strength['week'] = df_strength['start_date'].dt.to_period('W-SUN').apply(lambda r: r.end_time)
df_strength['time_min'] = df_strength['moving_time'] / 60

strength_text = fetch_text_fields(client, df_strength['id'].tolist())
df_strength = df_strength.merge(strength_text, on='id', how='left')
df_strength['volume_kg'] = df_strength.apply(
    lambda r: parse_volume_kg(r.get('description'), r.get('private_note')),
    axis=1,
)

# Drop early sessions where volume wasn't logged
df_strength = df_strength[df_strength['volume_kg'].notna()].copy()
df_strength['volume_kg'] = df_strength['volume_kg'].astype(int)
# Volume rate (kg lifted per minute) is the strength analogue of run/cycle avg speed:
# an effort-density metric, so colouring stays consistent with the other sports.
df_strength['volume_per_min'] = df_strength['volume_kg'] / df_strength['time_min']

# Cap the volume-rate colour scale (kg/min) so a couple of very dense sessions don't push
# every other bar/circle to one end of the colormap. Sessions above this clamp to max colour.
STRENGTH_COLOR_MAX = 350

print(f'{len(df_strength)} strength sessions with logged volume')
df_strength[['start_date', 'name', 'volume_kg', 'time_min', 'volume_per_min']].head()

## 14. Strength weekly stacked plot
Each bar = one week. Each stack segment = one session — height is `volume_kg`, color is `volume_per_min` (volume rate, kg lifted per minute). This mirrors the avg-speed coloring of the run/cycle plots: an effort-density metric rather than a raw magnitude.

In [ ]:
vis.plot_weekly_stacked(
    df_strength,
    stack_col='volume_kg',
    color_col='volume_per_min',
    stack_label='Volume (kg)',
    color_label='Volume rate (kg/min)',
    title='Weekly Strength Volume  |  Volume Rate',
    color_vmax=STRENGTH_COLOR_MAX,
    save_name='weekly_strength_volume.png',
)

## 15. Cycling — prepare data
Filter `Ride` activities from 2025+. Add `distance_km`, `week`, and `avg_speed_kmh` (Strava reports `average_speed` in m/s).


In [ ]:
df_rides = df_activities[
    (df_activities['type'] == 'Ride') &
    (df_activities['start_date'].dt.year >= 2025)
].copy()

df_rides['distance_km'] = df_rides['distance'] / 1000
df_rides['week'] = df_rides['start_date'].dt.to_period('W-SUN').apply(lambda r: r.end_time)
df_rides['avg_speed_kmh'] = df_rides['average_speed'] * 3.6

print(f'{len(df_rides)} rides from {df_rides["start_date"].min().date()} to {df_rides["start_date"].max().date()}')
df_rides[['start_date', 'name', 'distance_km', 'avg_speed_kmh']].head()


## 16. Cycling weekly stacked plot
Each bar = one week. Each stack segment = one ride — height is `distance_km`, color is `avg_speed_kmh`.


In [ ]:
vis.plot_weekly_stacked(
    df_rides,
    stack_col='distance_km',
    color_col='avg_speed_kmh',
    stack_label='Distance (km)',
    color_label='Avg speed (km/h)',
    title='Weekly Cycling Distance  |  Avg Speed',
    save_name='weekly_ride_speed.png',
)


## 17. All-sports weekly overview
Single figure stacking the weekly views for running, cycling, hiking, and strength on a shared time axis.
Slim per-panel height so everything fits without scrolling. For running, color = average speed (km/h).


In [ ]:
df_runs_overview = df_runs.copy()
df_runs_overview['avg_speed_kmh'] = df_runs_overview['average_speed'] * 3.6
# Trail runs share the legacy type 'Run'; sport_type tells them apart so they can be hatched.
df_runs_overview['is_trail'] = df_runs_overview['sport_type'].astype(str).str.contains('Trail', case=False, na=False)

panels = [
    dict(
        df=df_runs_overview,
        stack_col='distance_km', color_col='avg_speed_kmh',
        stack_label='Run km', color_label='km/h',
        title='Running  |  Avg Speed',
        hatch_col='is_trail',
    ),
    dict(
        df=df_rides,
        stack_col='distance_km', color_col='avg_speed_kmh',
        stack_label='Ride km', color_label='km/h',
        title='Cycling  |  Avg Speed',
    ),
    dict(
        df=df_hikes,
        stack_col='distance_km', color_col='weight_kg',
        stack_label='Hike km', color_label='kg',
        title='Hiking  |  Carried Weight',
    ),
    dict(
        df=df_strength,
        stack_col='volume_kg', color_col='volume_per_min',
        stack_label='Volume kg', color_label='kg/min',
        title='Strength  |  Volume Rate',
        color_vmax=STRENGTH_COLOR_MAX,
    ),
]

vis.plot_weekly_stacked_multi(
    panels,
    panel_height=2.5,
    save_name='weekly_overview_all_sports.png',
)

## 18. Monthly activity calendar (Strava-style)
Calendar grid where each day is a circle. Circle **color** reuses the same dark→white→orange metric colormap as the overview (run/bike avg speed, hike carried weight, strength volume rate kg/min), and circle **size** encodes the same magnitude that drives the overview bar heights (distance_km, or volume_kg for strength). The letter encodes sport: T=trail, R=run, H=hike, S=strength, B=bike. Extra activities on a day appear as smaller circles in the upper-right corner. Tune everything via `vis.CALENDAR` (min/max radius, `size_scaling` 'sqrt'/'linear', per-sport min/max values, etc.).

In [ ]:
import os
import matplotlib.colors as mcolors

cal_cmap = vis.metric_colormap()  # shared dark→white→orange metric colormap


def _metric_colors(values, vmin=None, vmax=None):
    """RGBA per value via the shared colormap, normalized within this sport's range.

    vmin / vmax override the data-derived range to cap the scale so a couple of extreme
    activities don't push every other circle to one end of the colormap; out-of-range
    values clamp to the end colors (clip=True).
    """
    v = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    finite = v[np.isfinite(v)]
    if finite.size == 0:
        return [cal_cmap(0.5)] * len(v)
    lo = float(finite.min()) if vmin is None else float(vmin)
    hi = float(finite.max()) if vmax is None else float(vmax)
    if hi <= lo:
        return [cal_cmap(0.5)] * len(v)
    norm = mcolors.Normalize(lo, hi, clip=True)
    return [cal_cmap(norm(x)) if np.isfinite(x) else cal_cmap(0.5) for x in v]


def _cal_date(row):
    d = row.get('start_date_local')
    if pd.isna(d):
        d = row['start_date']
    return pd.Timestamp(d).replace(tzinfo=None)


cal_rows = []

if len(df_runs) > 0:
    runs_cal = df_runs.copy()
    runs_cal['avg_speed_kmh'] = runs_cal['average_speed'] * 3.6
    is_trail = runs_cal['sport_type'].astype(str).str.contains('Trail', case=False, na=False)
    colors = _metric_colors(runs_cal['avg_speed_kmh'])
    for (_, row), color, trail in zip(runs_cal.iterrows(), colors, is_trail):
        cal_rows.append(dict(date=_cal_date(row), sport='trail' if trail else 'run',
                             size_value=row['distance_km'], color=color))

if len(df_rides) > 0:
    df_rides['avg_speed_kmh'] = df_rides['average_speed'] * 3.6
    colors = _metric_colors(df_rides['avg_speed_kmh'])
    for (_, row), color in zip(df_rides.iterrows(), colors):
        cal_rows.append(dict(date=_cal_date(row), sport='bike',
                             size_value=row['distance_km'], color=color))

if len(df_hikes) > 0:
    colors = _metric_colors(df_hikes['weight_kg'])
    for (_, row), color in zip(df_hikes.iterrows(), colors):
        cal_rows.append(dict(date=_cal_date(row), sport='hike',
                             size_value=row['distance_km'], color=color))

if len(df_strength) > 0:
    colors = _metric_colors(df_strength['volume_per_min'], vmax=STRENGTH_COLOR_MAX)
    for (_, row), color in zip(df_strength.iterrows(), colors):
        cal_rows.append(dict(date=_cal_date(row), sport='strength',
                             size_value=row['volume_kg'], color=color))

df_cal = pd.DataFrame(cal_rows)

if len(df_cal) > 0:
    df_cal['date'] = pd.to_datetime(df_cal['date'])
    months = sorted({(d.year, d.month) for d in df_cal['date']})  # oldest -> newest

    month_dir = os.path.join('plots', 'month_plots')
    os.makedirs(month_dir, exist_ok=True)

    # Index README at repo root, latest month on top. Images live in plots/month_plots/.
    lines = ["# Monthly Activity Calendars", "",
             "Auto-generated by `update_plots.py`. Latest month on top.", ""]
    for (y, m) in reversed(months):
        label = pd.Timestamp(year=y, month=m, day=1).strftime('%B %Y')
        lines += [f"### {label}", "", f"![{label}](plots/month_plots/{y}-{m:02d}.png)", ""]
    with open('CALENDAR_PLOTS.md', 'w') as f:
        f.write('\n'.join(lines).rstrip() + '\n')

    # Archive + display each non-empty month, latest -> oldest.
    for (y, m) in reversed(months):
        vis.plot_month_calendar(df_cal, year=y, month=m,
                                save_name=os.path.join('month_plots', f'{y}-{m:02d}.png'))